# Week 11: CNN — Conv2d, Pooling, and Training a Small CNN

**Topics:** `nn.Conv2d` (parameter count, output size, translation
equivariance) → `nn.MaxPool2d` → build and train a small CNN end-to-end on
STL-10 → data augmentation with `torchvision.transforms.v2` → Grad-CAM on
a pretrained model.

Week 10 trained the classifier but used a hand-crafted feature extractor
(BoVW). Today we **learn the feature extractor too** — the filters are
parameters, updated by the same `forward / backward / step` loop.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn

%matplotlib inline

### Environment setup

Works **locally** and on **Google Colab**. STL-10 is downloaded into a local
cache the first time you run it. We also install `grad-cam` for the final
Grad-CAM section.

In [ ]:
import os, subprocess, sys

IN_COLAB = "google.colab" in str(get_ipython()) if hasattr(__builtins__, "__IPYTHON__") else False

if IN_COLAB:
    DATA_DIR = "/content/stl10"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "grad-cam"])
else:
    DATA_DIR = os.path.expanduser("~/.cache/stl10")

os.makedirs(DATA_DIR, exist_ok=True)
print(f"Running on: {'Google Colab' if IN_COLAB else 'Local'}")
print(f"STL-10 cache: {DATA_DIR}")

### Display helpers

In [ ]:
def show_image(img, title=None, scale=4):
    """Display a single image. Grayscale (2-D) arrays use a gray colormap."""
    fig, ax = plt.subplots(figsize=(scale, scale))
    if img.ndim == 2:
        ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    else:
        ax.imshow(img)
    if title:
        ax.set_title(title)
    ax.axis("off")
    plt.tight_layout()
    plt.show()


def show_map(m, title=None, scale=4, cmap="viridis"):
    """Display a single 2-D float array (e.g., a feature map) with colormap."""
    fig, ax = plt.subplots(figsize=(scale, scale))
    im = ax.imshow(m, cmap=cmap)
    plt.colorbar(im, ax=ax, fraction=0.046)
    if title:
        ax.set_title(title)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

---

## 0. STL-10 — Same 5-Class Subset as Week 10

Same dataset, same five classes — but this time the network sees the **raw
pixels**, not BoVW vectors.

In [ ]:
from torchvision.datasets import STL10

train_ds = STL10(root=DATA_DIR, split="train", download=True)
test_ds = STL10(root=DATA_DIR, split="test", download=True)
print(f"len(train_ds) = {len(train_ds)}")
print(f"len(test_ds)  = {len(test_ds)}")

In [ ]:
SELECTED = [0, 1, 2, 3, 8]
PER_CLASS_TRAIN = 200
PER_CLASS_TEST = 30

class_names = []
for c in SELECTED:
    class_names.append(train_ds.classes[c])


def select_subset(ds, n_per_class):
    """Keep n_per_class images from each SELECTED class; remap labels to 0..N-1."""
    new_label = {}
    for new_idx, old_label in enumerate(SELECTED):
        new_label[old_label] = new_idx
    counts = {}
    for c in SELECTED:
        counts[c] = 0
    items = []
    for i in range(len(ds)):
        img, label = ds[i]
        if label not in new_label:
            continue
        if counts[label] >= n_per_class:
            continue
        gray = np.array(img.convert("L"))
        items.append((gray, new_label[label]))
        counts[label] += 1
    return items


train_items = select_subset(train_ds, PER_CLASS_TRAIN)
test_items = select_subset(test_ds, PER_CLASS_TEST)
print(f"class_names      = {class_names}")
print(f"len(train_items) = {len(train_items)}")
print(f"len(test_items)  = {len(test_items)}")

### Stack into image tensors

For the CNN we want **(N, C, H, W)** float tensors — channel-first. Grayscale
has C = 1. Pixel values are normalized to [0, 1] by dividing by 255.

In [ ]:
N_train = len(train_items)
N_test = len(test_items)

X_train_t = torch.zeros((N_train, 1, 96, 96), dtype=torch.float32)
y_train_t = torch.zeros(N_train, dtype=torch.long)
for i in range(N_train):
    gray, lab = train_items[i]
    X_train_t[i, 0] = torch.from_numpy(gray).float() / 255.0
    y_train_t[i] = lab

X_test_t = torch.zeros((N_test, 1, 96, 96), dtype=torch.float32)
y_test_t = torch.zeros(N_test, dtype=torch.long)
for i in range(N_test):
    gray, lab = test_items[i]
    X_test_t[i, 0] = torch.from_numpy(gray).float() / 255.0
    y_test_t[i] = lab

print(f"X_train_t.shape = {tuple(X_train_t.shape)}, dtype = {X_train_t.dtype}")
print(f"y_train_t.shape = {tuple(y_train_t.shape)}, dtype = {y_train_t.dtype}")

---

## 1. `nn.Conv2d` — A Learnable Filter

`nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)` is **the**
key function of this week. Everything else (output size, parameter count,
translation equivariance) follows from understanding its five arguments.

### 1.1 Define a Conv2d layer and inspect it

A Conv2d layer with $C_\text{in} \to C_\text{out}$ filters of size $K \times K$ has

$$\#\text{params} = K \cdot K \cdot C_\text{in} \cdot C_\text{out} + C_\text{out}$$

`conv.weight` has shape `(C_out, C_in, K, K)` and `conv.bias` has shape `(C_out,)`.

In [ ]:
conv = nn.Conv2d(in_channels=1, out_channels=4, kernel_size=3, padding=1)

print(f"conv.weight.shape = {tuple(conv.weight.shape)}   # (C_out, C_in, K, K)")
print(f"conv.bias.shape   = {tuple(conv.bias.shape)}      # (C_out,)")

total = sum(p.numel() for p in conv.parameters())
print(f"total parameters  = {total}")
print(f"formula           = 3*3*1*4 + 4 = {3*3*1*4 + 4}")

### 1.2 Apply Conv2d to one image

Conv2d expects input of shape `(N, C_in, H, W)`. The output has shape
`(N, C_out, H_out, W_out)` — **one feature map per output channel**.

In [ ]:
x = X_train_t[0:1]                     # one image, with batch dim: (1, 1, 96, 96)
out = conv(x)
print(f"x.shape   = {tuple(x.shape)}")
print(f"out.shape = {tuple(out.shape)}   # (N, C_out, H, W)")

In [ ]:
show_image((x[0, 0].numpy() * 255).astype(np.uint8), title="input")

for k in range(4):
    fmap = out[0, k].detach().numpy()
    show_map(fmap, title=f"feature map {k} (random filter)")

### 1.3 Output size — vary K, S, P

$$H_\text{out} = \left\lfloor \frac{H_\text{in} + 2P - K}{S} \right\rfloor + 1$$

Four variations on the same 96×96 input to verify the formula by experiment.

In [ ]:
x = X_train_t[0:1]   # (1, 1, 96, 96)

conv_same    = nn.Conv2d(1, 1, kernel_size=3, stride=1, padding=1)
conv_valid   = nn.Conv2d(1, 1, kernel_size=3, stride=1, padding=0)
conv_stride2 = nn.Conv2d(1, 1, kernel_size=3, stride=2, padding=1)
conv_big     = nn.Conv2d(1, 1, kernel_size=7, stride=2, padding=3)

print(f"K=3 S=1 P=1 -> {tuple(conv_same(x).shape)}    # 'same' padding")
print(f"K=3 S=1 P=0 -> {tuple(conv_valid(x).shape)}    # 'valid' padding, shrinks by K-1")
print(f"K=3 S=2 P=1 -> {tuple(conv_stride2(x).shape)}    # stride-2 halves H, W")
print(f"K=7 S=2 P=3 -> {tuple(conv_big(x).shape)}    # ResNet's first layer pattern")

### 1.4 Translation equivariance — set the filter by hand

So far the filter has been random. To **see** what a filter does, we'll
manually set Conv2d's weights to a Sobel-like horizontal-edge filter, run
it on an image, then shift the input by 20 pixels and run it again. The
output should shift by the **same** amount — that's translation
equivariance, which comes for free from parameter sharing.

In [ ]:
sobel_x = torch.tensor([[-1.0, 0.0, 1.0],
                        [-2.0, 0.0, 2.0],
                        [-1.0, 0.0, 1.0]])

edge_conv = nn.Conv2d(1, 1, kernel_size=3, padding=1, bias=False)
with torch.no_grad():
    edge_conv.weight[0, 0] = sobel_x

print(f"edge_conv.weight[0, 0] =\n{edge_conv.weight[0, 0]}")

In [ ]:
img = X_train_t[0:1]            # (1, 1, 96, 96)

# Shift the input 20 pixels to the right
img_shifted = torch.zeros_like(img)
img_shifted[0, 0, :, 20:] = img[0, 0, :, :-20]

with torch.no_grad():
    out = edge_conv(img)
    out_shifted = edge_conv(img_shifted)

show_image((img[0, 0].numpy() * 255).astype(np.uint8), title="input")
show_map(out[0, 0].numpy(), title="edge_conv(input)")
show_image((img_shifted[0, 0].numpy() * 255).astype(np.uint8), title="input shifted by 20px")
show_map(out_shifted[0, 0].numpy(), title="edge_conv(shifted) — same edges, shifted")

---

## 2. `nn.MaxPool2d` — Downsample Without Parameters

`MaxPool2d` slides a window across the feature map and keeps the **max** in
each window. Same output-size formula as Conv2d — but **no learnable
parameters**.

### 2.1 Halve H, W with a 2×2 max-pool

In [ ]:
pool = nn.MaxPool2d(kernel_size=2, stride=2)

n_params = sum(p.numel() for p in pool.parameters())
print(f"pool parameters = {n_params}")

x = X_train_t[0:1]              # (1, 1, 96, 96)
out = pool(x)
print(f"x.shape   = {tuple(x.shape)}")
print(f"out.shape = {tuple(out.shape)}   # H, W halved; C unchanged")

### 2.2 Max vs Average pool, side by side

`MaxPool` keeps the **strongest activation** in each window; `AvgPool`
keeps the **mean**. On natural images, max-pool sharpens; avg-pool blurs.

In [ ]:
max_pool = nn.MaxPool2d(kernel_size=4, stride=4)
avg_pool = nn.AvgPool2d(kernel_size=4, stride=4)

x = X_train_t[0:1]
max_out = max_pool(x)
avg_out = avg_pool(x)

show_image((x[0, 0].numpy() * 255).astype(np.uint8), title="input (96×96)")
show_image((max_out[0, 0].numpy() * 255).astype(np.uint8), title="max-pool 4× (24×24)")
show_image((avg_out[0, 0].numpy() * 255).astype(np.uint8), title="avg-pool 4× (24×24)")

---

## 3. Build & Train a Small CNN

A CNN body is **Conv → ReLU → Pool** blocks stacked: spatial dimensions
shrink, channel dimension grows. A small FC head at the end maps the final
feature volume to class scores. Trained end-to-end with the same
`forward / backward / step` loop from week 10.

### 3.1 Define the CNN

Three Conv→ReLU→Pool blocks (1 → 16 → 32 → 64 channels), then Flatten → FC
→ 5 class scores. After three 2×2 pools, the 96×96 image becomes 12×12.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()
        self.fc = nn.Linear(64 * 12 * 12, 5)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))   # 96 -> 48
        x = self.pool(self.relu(self.conv2(x)))   # 48 -> 24
        x = self.pool(self.relu(self.conv3(x)))   # 24 -> 12
        x = x.flatten(start_dim=1)                # (N, 64*12*12)
        x = self.fc(x)
        return x


cnn = SmallCNN()
print(cnn)
print(f"\nCNN parameters: {sum(p.numel() for p in cnn.parameters()):,}")

In [ ]:
# One untrained forward pass to verify shapes
out = cnn(X_train_t[:4])
print(f"out.shape = {tuple(out.shape)}   # (batch, num_classes)")

### 3.2 Train with the week-10 ritual — `forward / backward / step`

Same four-line loop from week 10. Only the input changed: feature vectors
became image tensors.

In [ ]:
from torch import optim


def train_model(model, X, y, lr, epochs):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    losses = []
    for epoch in range(epochs):
        logits = model(X)
        loss = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses


def test_accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        pred = model(X).argmax(dim=1)
    model.train()
    return (pred == y).float().mean().item()

In [ ]:
torch.manual_seed(0)
cnn = SmallCNN()
cnn_losses = train_model(cnn, X_train_t, y_train_t, lr=1e-3, epochs=30)
cnn_acc = test_accuracy(cnn, X_test_t, y_test_t)
print(f"CNN test accuracy = {cnn_acc:.3f}")

### 3.3 Why not just flatten and use an MLP?

For comparison, train an MLP on the **flattened raw pixels** — 96·96 = 9216
input features. Same number of hidden units as the week-10 MLP.

In [ ]:
class FlatMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(96 * 96, 32)
        self.fc2 = nn.Linear(32, 5)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.flatten(start_dim=1)
        x = self.relu(self.fc1(x))
        return self.fc2(x)


torch.manual_seed(0)
mlp = FlatMLP()
print(f"MLP parameters: {sum(p.numel() for p in mlp.parameters()):,}")
print(f"CNN parameters: {sum(p.numel() for p in cnn.parameters()):,}")

mlp_losses = train_model(mlp, X_train_t, y_train_t, lr=1e-3, epochs=30)
mlp_acc = test_accuracy(mlp, X_test_t, y_test_t)
print(f"MLP test accuracy = {mlp_acc:.3f}")

In [ ]:
plt.figure(figsize=(6, 3))
plt.plot(cnn_losses, label=f"CNN  (acc {cnn_acc:.2f})")
plt.plot(mlp_losses, label=f"MLP  (acc {mlp_acc:.2f})")
plt.xlabel("epoch")
plt.ylabel("training loss")
plt.legend()
plt.tight_layout()
plt.show()

### Exercise 3.1 — Tweak the architecture

Modify `SmallCNN` and retrain it. Pick **one** change (or combine them),
train with `train_model(...)`, and compare test accuracy.

Suggestions:
- Add a fourth Conv→ReLU→Pool block (note: spatial would become 6×6 after 4 pools — update `fc`)
- Double the channel widths (e.g., 1 → 32 → 64 → 128)
- Use larger kernels (e.g., 5×5) with matching padding

**Variables to create:** `new_cnn`, `new_losses`, `new_acc`.

In [ ]:
# YOUR CODE HERE
# Define a modified CNN class, train it with train_model(...), and compute
# its test accuracy.
# Variables to create: new_cnn, new_losses, new_acc


# Comparison plot (uses your variables)
plt.figure(figsize=(6, 3))
plt.plot(cnn_losses, label=f"baseline CNN  (acc {cnn_acc:.2f})")
plt.plot(new_losses, label=f"modified CNN  (acc {new_acc:.2f})")
plt.xlabel("epoch")
plt.ylabel("training loss")
plt.legend()
plt.tight_layout()
plt.show()

---

## 4. Data Augmentation with `torchvision.transforms.v2`

`transforms.v2` provides composable image transforms. We apply them to one
image to **see** what each one does, then chain them together.

In [ ]:
from torchvision.transforms import v2

# Pick one image from train_ds — keep it as a PIL.Image for transforms.v2
img_pil, label = train_ds[0]
print(f"image size = {img_pil.size}, label = {train_ds.classes[label]}")
show_image(np.array(img_pil), title=f"original — {train_ds.classes[label]}")

In [ ]:
flip = v2.RandomHorizontalFlip(p=1.0)
crop = v2.RandomResizedCrop(size=96, scale=(0.5, 0.9))
jitter = v2.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5)
rotate = v2.RandomRotation(degrees=20)

show_image(np.array(flip(img_pil)), title="RandomHorizontalFlip")
show_image(np.array(crop(img_pil)), title="RandomResizedCrop")
show_image(np.array(jitter(img_pil)), title="ColorJitter")
show_image(np.array(rotate(img_pil)), title="RandomRotation")

### Chain them together

Real training composes several augmentations. Each draw from the chain is
a fresh, plausible variation of the source image — effectively expanding
the training set on the fly.

In [ ]:
augment = v2.Compose([
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomResizedCrop(size=96, scale=(0.6, 1.0)),
    v2.ColorJitter(brightness=0.3, contrast=0.3),
])

torch.manual_seed(0)
for k in range(4):
    show_image(np.array(augment(img_pil)), title=f"augmented sample {k}")

---

## 5. Grad-CAM — Where Is the CNN Looking?

A trained CNN's confidence "this is a cat" is only one number. **Grad-CAM**
turns that number into a heatmap on the input — which pixels drove the
prediction.

We use a **pretrained** `resnet18` (ImageNet, which contains all five of
our STL-10 classes) so we have a strong classifier without training. The
`grad-cam` library handles the hook plumbing for us.

### 5.1 Load pretrained ResNet-18 and pick an image

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

weights = ResNet18_Weights.IMAGENET1K_V1
resnet = resnet18(weights=weights)
resnet.eval()

preprocess = weights.transforms()
print(preprocess)

In [ ]:
# Find a clearly-identifiable cat image in STL-10
cat_idx = None
for i in range(len(train_ds)):
    img, lbl = train_ds[i]
    if train_ds.classes[lbl] == "cat":
        cat_idx = i
        break

img_pil, label = train_ds[cat_idx]
show_image(np.array(img_pil), title=f"STL-10 — {train_ds.classes[label]}")

# Preprocess for ResNet (resize + normalize) and add batch dim
input_tensor = preprocess(img_pil).unsqueeze(0)
print(f"input_tensor.shape = {tuple(input_tensor.shape)}")

with torch.no_grad():
    logits = resnet(input_tensor)
top5 = torch.topk(logits[0], k=5)
for i in range(5):
    cls_idx = top5.indices[i].item()
    score = top5.values[i].item()
    print(f"  #{i+1}: {weights.meta['categories'][cls_idx]:<30s}  score = {score:.2f}")

### 5.2 Grad-CAM heatmap

`GradCAM(model, target_layers=[...])` registers hooks on the chosen conv
layer, runs forward + backward, and returns a normalized heatmap. We target
**`resnet.layer4[-1]`** — ResNet-18's last conv block — so the heatmap has
the largest receptive field and the strongest class signal.

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

cam = GradCAM(model=resnet, target_layers=[resnet.layer4[-1]])

grayscale_cam = cam(input_tensor=input_tensor)
grayscale_cam = grayscale_cam[0]   # (H, W) float in [0, 1]
print(f"grayscale_cam.shape = {grayscale_cam.shape}, range = [{grayscale_cam.min():.2f}, {grayscale_cam.max():.2f}]")

# Build the [0, 1] RGB image that show_cam_on_image expects
rgb_for_overlay = np.array(img_pil.resize((224, 224))).astype(np.float32) / 255.0
overlay = show_cam_on_image(rgb_for_overlay, grayscale_cam, use_rgb=True)

show_map(grayscale_cam, title="Grad-CAM heatmap", cmap="jet")
show_image(overlay, title="Grad-CAM overlay")

---

## Recap

- **`nn.Conv2d(C_in, C_out, K, S, P)`** — five arguments determine
  output size and parameter count. Same filter is reused at every position
  (parameter sharing → translation equivariance).
- **`nn.MaxPool2d`** — downsamples spatial dims with **no learnable
  parameters**.
- **A small CNN** trained on the same 5-class STL-10 subset beats an MLP
  on flattened pixels — fewer parameters, better inductive bias.
- **`transforms.v2`** — composable, on-the-fly augmentation for
  regularization and invariance.
- **Grad-CAM** — one forward pass + one partial backward pass tell you
  where the CNN is looking.